# Lista de Exercícios — Sistemas Lineares
## Eliminação de Gauss e modelo de balanço térmico

**Disciplina:** Cálculo Numérico — Profa. Raquel J. Lobosco (UNIFESP)
**Aluno:** Matheus Bueno Christovão — RA 185302

Este notebook reúne a resolução da lista. As resoluções manuais estão nas células de texto, e o código
reproduz e confere cada passo. Todas as resoluções usam uma única função, `eliminacao_gauss`, implementada **sem pivoteamento**.

| Exercício | Tema |
|---|---|
| 1 | Sistemas na forma matricial $A\mathbf{x} = \mathbf{b}$ |
| 2 | Eliminação de Gauss passo a passo |
| 3 | Número de soluções (Rouché–Capelli) |
| 4 | Implementação própria × `numpy.linalg.solve` |
| 5 | Modelo de balanço térmico |

## Preparação

In [1]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)


def titulo(texto):
    print("\n" + "=" * 70)
    print(texto)
    print("=" * 70)

## Função principal: eliminação de Gauss sem pivoteamento

A função segue as quatro etapas do método:

1. monta a matriz aumentada $[A \mid \mathbf{b}]$;
2. faz a **eliminação progressiva**: para cada pivô $a_{kk}$, calcula os multiplicadores $m_{ik} = a_{ik}/a_{kk}$ e aplica $L_i \leftarrow L_i - m_{ik}L_k$;
3. faz a **substituição regressiva**: $x_i = \dfrac{c_i - \sum_{j>i} u_{ij}x_j}{u_{ii}}$;
4. exibe a solução.

Sem pivoteamento, um pivô nulo interrompe o método, e por isso a função verifica cada pivô antes de dividir.

In [2]:
def eliminacao_gauss(A, b, verbose=True, tol=1e-12):
    """
    Resolve Ax = b por eliminação de Gauss SEM pivoteamento.

    Parâmetros
    ----------
    A : matriz n x n dos coeficientes
    b : vetor independente (n)
    verbose : se True, imprime a matriz aumentada a cada etapa
    tol : tolerância para considerar um pivô como nulo

    Retorna
    -------
    x : vetor solução
    M : matriz aumentada triangular superior final [U | c]
    """
    # Conversão para float (evita divisão inteira e não altera os dados originais)
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    n = len(b)

    if A.shape != (n, n):
        raise ValueError("A deve ser quadrada e compatível com o tamanho de b.")

    # 1) Formação da matriz aumentada [A | b]
    M = np.hstack([A, b.reshape(-1, 1)])
    if verbose:
        print("Matriz aumentada inicial [A | b]:")
        print(M)

    # 2) Eliminação progressiva: zera os elementos abaixo de cada pivô M[k, k]
    for k in range(n - 1):
        # Sem pivoteamento, um pivô nulo interrompe o método
        if abs(M[k, k]) < tol:
            raise ZeroDivisionError(
                f"Pivô nulo na posição ({k+1},{k+1}). "
                "Sem pivoteamento, o método não pode prosseguir."
            )
        for i in range(k + 1, n):
            m = M[i, k] / M[k, k]           # multiplicador m_ik
            M[i, k:] = M[i, k:] - m * M[k, k:]  # L_i <- L_i - m_ik * L_k
            if verbose:
                print(f"  L{i+1} <- L{i+1} - ({m:.4g}) * L{k+1}")
        if verbose:
            print(f"Matriz após zerar a coluna {k+1}:")
            print(M)

    if abs(M[n - 1, n - 1]) < tol:
        raise ZeroDivisionError("Último pivô nulo: matriz singular (sem solução única).")

    # 3) Substituição regressiva: resolve Ux = c de baixo para cima
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        soma = M[i, i + 1:n] @ x[i + 1:n]   # soma dos termos já conhecidos
        x[i] = (M[i, n] - soma) / M[i, i]

    # 4) Exibição da solução
    if verbose:
        print("Solução por substituição regressiva:")
        for i, xi in enumerate(x):
            print(f"  x{i+1} = {xi:.6f}")

    return x, M

A função auxiliar abaixo compara a solução com `numpy.linalg.solve` e calcula o resíduo $\lVert A\mathbf{x} - \mathbf{b} \rVert$.

In [3]:
def comparar_com_numpy(A, b, x_gauss):
    """Compara a solução própria com numpy.linalg.solve e mostra o resíduo."""
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    x_np = np.linalg.solve(A, b)
    print("numpy.linalg.solve      :", x_np)
    print("Eliminação de Gauss     :", x_gauss)
    print("||x_gauss - x_numpy||   :", np.linalg.norm(x_gauss - x_np))
    print("Resíduo ||A x - b||     :", np.linalg.norm(A @ x_gauss - b))
    print("Soluções coincidem?     :", np.allclose(x_gauss, x_np))

## Exercício 1 — Sistemas lineares na forma matricial

**a)**
$$\begin{cases} 2x + y = 7 \\ x - 3y = -5 \end{cases}
\quad\Longrightarrow\quad
\underbrace{\begin{bmatrix} 2 & 1 \\ 1 & -3 \end{bmatrix}}_{A}
\underbrace{\begin{bmatrix} x \\ y \end{bmatrix}}_{\mathbf{x}} =
\underbrace{\begin{bmatrix} 7 \\ -5 \end{bmatrix}}_{\mathbf{b}}$$

**b)**
$$\begin{cases} x + 2y - z = 2 \\ 3x - y + 2z = 9 \\ 2x + y + z = 7 \end{cases}
\quad\Longrightarrow\quad
\begin{bmatrix} 1 & 2 & -1 \\ 3 & -1 & 2 \\ 2 & 1 & 1 \end{bmatrix}
\begin{bmatrix} x \\ y \\ z \end{bmatrix} =
\begin{bmatrix} 2 \\ 9 \\ 7 \end{bmatrix}$$

In [4]:
def exercicio_1():
    titulo("EXERCÍCIO 1 — Forma matricial Ax = b")

    print("a) A =\n", np.array([[2, 1], [1, -3]]))
    print("   x = [x, y]^T")
    print("   b =", np.array([7, -5]))
    print("   Solução (verificação):", np.linalg.solve([[2, 1], [1, -3]], [7, -5]))

    print("\nb) A =\n", np.array([[1, 2, -1], [3, -1, 2], [2, 1, 1]]))
    print("   x = [x, y, z]^T")
    print("   b =", np.array([2, 9, 7]))
    print("   det(A) =", round(np.linalg.det([[1, 2, -1], [3, -1, 2], [2, 1, 1]]), 6))
    print("   Solução (verificação):",
          np.linalg.solve([[1, 2, -1], [3, -1, 2], [2, 1, 1]], [2, 9, 7]))


exercicio_1()


EXERCÍCIO 1 — Forma matricial Ax = b
a) A =
 [[ 2  1]
 [ 1 -3]]
   x = [x, y]^T
   b = [ 7 -5]
   Solução (verificação): [2.2857 2.4286]

b) A =
 [[ 1  2 -1]
 [ 3 -1  2]
 [ 2  1  1]]
   x = [x, y, z]^T
   b = [2 9 7]
   det(A) = -6.0
   Solução (verificação): [2. 1. 2.]


## Exercício 2 — Eliminação de Gauss (resolução manual)

$$\begin{cases} 2x + y - z = 8 \\ -3x - y + 2z = -11 \\ -2x + y + 2z = -3 \end{cases}
\qquad
[A \mid \mathbf{b}] = \left[\begin{array}{ccc|c} 2 & 1 & -1 & 8 \\ -3 & -1 & 2 & -11 \\ -2 & 1 & 2 & -3 \end{array}\right]$$

**Etapa 1 — pivô $a_{11} = 2$**

- $m_{21} = -3/2 = -1{,}5 \;\Rightarrow\; L_2 \leftarrow L_2 + 1{,}5\,L_1 = (0;\ 0{,}5;\ 0{,}5;\ 1)$
- $m_{31} = -2/2 = -1 \;\Rightarrow\; L_3 \leftarrow L_3 + L_1 = (0;\ 2;\ 1;\ 5)$

$$\left[\begin{array}{ccc|c} 2 & 1 & -1 & 8 \\ 0 & 0{,}5 & 0{,}5 & 1 \\ 0 & 2 & 1 & 5 \end{array}\right]$$

**Etapa 2 — pivô $a_{22} = 0{,}5$**

- $m_{32} = 2/0{,}5 = 4 \;\Rightarrow\; L_3 \leftarrow L_3 - 4L_2 = (0;\ 0;\ -1;\ 1)$

$$\left[\begin{array}{ccc|c} 2 & 1 & -1 & 8 \\ 0 & 0{,}5 & 0{,}5 & 1 \\ 0 & 0 & -1 & 1 \end{array}\right] \quad \text{(triangular superior)}$$

**Substituição regressiva**

- $L_3$: $-z = 1 \Rightarrow z = -1$
- $L_2$: $0{,}5y + 0{,}5(-1) = 1 \Rightarrow y = 3$
- $L_1$: $2x + 3 - (-1) = 8 \Rightarrow x = 2$

**Solução:** $(x, y, z) = (2,\ 3,\ -1)$

O código abaixo reproduz os mesmos passos e imprime os multiplicadores.

In [5]:
A2 = [[2, 1, -1],
      [-3, -1, 2],
      [-2, 1, 2]]
b2 = [8, -11, -3]


def exercicio_2():
    titulo("EXERCÍCIO 2 — Eliminação de Gauss (passos reproduzidos no código)")
    x, _ = eliminacao_gauss(A2, b2, verbose=True)
    print("\nResultado esperado (manual): x = 2, y = 3, z = -1")
    return x


exercicio_2()


EXERCÍCIO 2 — Eliminação de Gauss (passos reproduzidos no código)
Matriz aumentada inicial [A | b]:
[[  2.   1.  -1.   8.]
 [ -3.  -1.   2. -11.]
 [ -2.   1.   2.  -3.]]
  L2 <- L2 - (-1.5) * L1
  L3 <- L3 - (-1) * L1
Matriz após zerar a coluna 1:
[[ 2.   1.  -1.   8. ]
 [ 0.   0.5  0.5  1. ]
 [ 0.   2.   1.   5. ]]
  L3 <- L3 - (4) * L2
Matriz após zerar a coluna 2:
[[ 2.   1.  -1.   8. ]
 [ 0.   0.5  0.5  1. ]
 [ 0.   0.  -1.   1. ]]
Solução por substituição regressiva:
  x1 = 2.000000
  x2 = 3.000000
  x3 = -1.000000

Resultado esperado (manual): x = 2, y = 3, z = -1


array([ 2.,  3., -1.])

## Exercício 3 — Número de soluções

Critério de Rouché–Capelli, com $n$ = número de incógnitas:

| Condição | Classificação |
|---|---|
| $\text{posto}(A) = \text{posto}([A \mid \mathbf{b}]) = n$ | solução única (SPD) |
| $\text{posto}(A) = \text{posto}([A \mid \mathbf{b}]) < n$ | infinitas soluções (SPI) |
| $\text{posto}(A) < \text{posto}([A \mid \mathbf{b}])$ | nenhuma solução (SI) |

**a)** $\left[\begin{array}{cc|c} 1 & 1 & 3 \\ 2 & 2 & 6 \end{array}\right] \xrightarrow{L_2 - 2L_1} \left[\begin{array}{cc|c} 1 & 1 & 3 \\ 0 & 0 & 0 \end{array}\right]$
A linha $0 = 0$ é sempre verdadeira e sobra uma equação para duas incógnitas: posto $= 1 < 2$, **infinitas soluções**.
Solução geral: $y = t,\ x = 3 - t\ (t \in \mathbb{R})$. Retas coincidentes.

**b)** $\left[\begin{array}{cc|c} 1 & 1 & 3 \\ 2 & 2 & 7 \end{array}\right] \xrightarrow{L_2 - 2L_1} \left[\begin{array}{cc|c} 1 & 1 & 3 \\ 0 & 0 & 1 \end{array}\right]$
A linha diz $0 = 1$, o que é absurdo: posto$(A) = 1 <$ posto$([A \mid \mathbf{b}]) = 2$, **nenhuma solução**. Retas paralelas distintas.

**c)** $\left[\begin{array}{cc|c} 1 & 1 & 3 \\ 2 & -1 & 0 \end{array}\right] \xrightarrow{L_2 - 2L_1} \left[\begin{array}{cc|c} 1 & 1 & 3 \\ 0 & -3 & -6 \end{array}\right]$
$-3y = -6 \Rightarrow y = 2$ e $x + 2 = 3 \Rightarrow x = 1$: posto $= 2$, **solução única** $(x, y) = (1, 2)$. Retas concorrentes.

In [6]:
def classificar_sistema(A, b):
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float).reshape(-1, 1)
    n = A.shape[1]
    posto_A = np.linalg.matrix_rank(A)
    posto_Ab = np.linalg.matrix_rank(np.hstack([A, b]))
    if posto_A < posto_Ab:
        tipo = "nenhuma solução (sistema impossível)"
    elif posto_A < n:
        tipo = "infinitas soluções (sistema possível e indeterminado)"
    else:
        tipo = "solução única (sistema possível e determinado)"
    return posto_A, posto_Ab, tipo


def exercicio_3():
    titulo("EXERCÍCIO 3 — Número de soluções")
    sistemas = {
        "a": ([[1, 1], [2, 2]], [3, 6]),
        "b": ([[1, 1], [2, 2]], [3, 7]),
        "c": ([[1, 1], [2, -1]], [3, 0]),
    }
    for item, (A, b) in sistemas.items():
        pA, pAb, tipo = classificar_sistema(A, b)
        print(f"{item}) posto(A) = {pA}, posto([A|b]) = {pAb}  ->  {tipo}")
        if pA == pAb == len(b):
            print("   Solução:", np.linalg.solve(A, b))


exercicio_3()


EXERCÍCIO 3 — Número de soluções
a) posto(A) = 1, posto([A|b]) = 1  ->  infinitas soluções (sistema possível e indeterminado)
b) posto(A) = 1, posto([A|b]) = 2  ->  nenhuma solução (sistema impossível)
c) posto(A) = 2, posto([A|b]) = 2  ->  solução única (sistema possível e determinado)
   Solução: [1. 2.]


## Exercício 4 — Implementação da eliminação de Gauss e comparação com NumPy

A implementação é a função `eliminacao_gauss`, definida no início do notebook. Ela recebe $A$ e $\mathbf{b}$,
forma a matriz aumentada, elimina os elementos abaixo dos pivôs, aplica a substituição regressiva e exibe a solução.

**Conclusão:** as duas soluções coincidem até a precisão de máquina (diferença da ordem de $10^{-15}$ ou nula) e o
resíduo $\lVert A\mathbf{x} - \mathbf{b} \rVert$ é praticamente nulo. Sem pivoteamento o método funciona aqui porque
nenhum pivô se anula ($2$, $0{,}5$ e $-1$). Em geral, pivôs nulos ou muito pequenos causam falha ou amplificação de erros
de arredondamento. Por isso existe o pivoteamento parcial, que o `numpy.linalg.solve` usa internamente (via LAPACK, fatoração LU).

In [7]:
def exercicio_4():
    titulo("EXERCÍCIO 4 — Gauss sem pivoteamento vs numpy.linalg.solve")
    x, _ = eliminacao_gauss(A2, b2, verbose=False)
    comparar_com_numpy(A2, b2, x)


exercicio_4()


EXERCÍCIO 4 — Gauss sem pivoteamento vs numpy.linalg.solve
numpy.linalg.solve      : [ 2.  3. -1.]
Eliminação de Gauss     : [ 2.  3. -1.]
||x_gauss - x_numpy||   : 1.3732700395566711e-15
Resíduo ||A x - b||     : 0.0
Soluções coincidem?     : True


## Exercício 5 — Modelo de balanço térmico

$$\begin{bmatrix} 4 & -1 & 0 \\ -1 & 4 & -1 \\ 0 & -1 & 3 \end{bmatrix}
\begin{bmatrix} T_1 \\ T_2 \\ T_3 \end{bmatrix} =
\begin{bmatrix} 15 \\ 10 \\ 10 \end{bmatrix}$$

### a) Resolução manual

**Etapa 1 — pivô $a_{11} = 4$**

- $m_{21} = -1/4 \;\Rightarrow\; L_2 \leftarrow L_2 + \tfrac14 L_1 = \left(0;\ \tfrac{15}{4};\ -1;\ \tfrac{55}{4}\right)$
- $m_{31} = 0/4 = 0 \;\Rightarrow\; L_3$ inalterada

$$\left[\begin{array}{ccc|c} 4 & -1 & 0 & 15 \\ 0 & 15/4 & -1 & 55/4 \\ 0 & -1 & 3 & 10 \end{array}\right]$$

**Etapa 2 — pivô $a_{22} = 15/4$**

- $m_{32} = \dfrac{-1}{15/4} = -\dfrac{4}{15} \;\Rightarrow\; L_3 \leftarrow L_3 + \tfrac{4}{15}L_2$
- $a_{33} = 3 - \tfrac{4}{15} = \tfrac{41}{15} \approx 2{,}7333$ e $b_3 = 10 + \tfrac{4}{15}\cdot\tfrac{55}{4} = \tfrac{41}{3} \approx 13{,}6667$

$$\left[\begin{array}{ccc|c} 4 & -1 & 0 & 15 \\ 0 & 15/4 & -1 & 55/4 \\ 0 & 0 & 41/15 & 41/3 \end{array}\right]$$

**Substituição regressiva**

- $T_3 = \dfrac{41/3}{41/15} = 5$
- $\tfrac{15}{4}T_2 - 5 = \tfrac{55}{4} \Rightarrow T_2 = 5$
- $4T_1 - 5 = 15 \Rightarrow T_1 = 5$

**Solução:** $T_1 = T_2 = T_3 = 5$

**Interpretação:** a rede entra em equilíbrio com as três temperaturas iguais (5 unidades). A matriz é **tridiagonal,
simétrica e diagonalmente dominante** ($|4| > |-1| + |0|$, $|4| > |-1| + |-1|$, $|3| > |-1| + |0|$), o que garante pivôs
não nulos na eliminação sem pivoteamento e boa estabilidade numérica. A estrutura é a mesma de uma análise nodal em
circuitos resistivos (condutâncias na matriz, "fontes" no vetor independente).

### b) Implementação em Python

In [8]:
A5 = [[4, -1, 0],
      [-1, 4, -1],
      [0, -1, 3]]
b5 = [15, 10, 10]


def exercicio_5():
    titulo("EXERCÍCIO 5 — Modelo de balanço térmico")
    T, _ = eliminacao_gauss(A5, b5, verbose=True)
    print()
    comparar_com_numpy(A5, b5, T)
    print(f"\nTemperaturas: T1 = {T[0]:.4f}, T2 = {T[1]:.4f}, T3 = {T[2]:.4f}")
    print("Resultado esperado (manual): T1 = T2 = T3 = 5")


exercicio_5()


EXERCÍCIO 5 — Modelo de balanço térmico
Matriz aumentada inicial [A | b]:
[[ 4. -1.  0. 15.]
 [-1.  4. -1. 10.]
 [ 0. -1.  3. 10.]]
  L2 <- L2 - (-0.25) * L1
  L3 <- L3 - (0) * L1
Matriz após zerar a coluna 1:
[[ 4.   -1.    0.   15.  ]
 [ 0.    3.75 -1.   13.75]
 [ 0.   -1.    3.   10.  ]]
  L3 <- L3 - (-0.2667) * L2
Matriz após zerar a coluna 2:
[[ 4.     -1.      0.     15.    ]
 [ 0.      3.75   -1.     13.75  ]
 [ 0.      0.      2.7333 13.6667]]
Solução por substituição regressiva:
  x1 = 5.000000
  x2 = 5.000000
  x3 = 5.000000

numpy.linalg.solve      : [5. 5. 5.]
Eliminação de Gauss     : [5. 5. 5.]
||x_gauss - x_numpy||   : 0.0
Resíduo ||A x - b||     : 0.0
Soluções coincidem?     : True

Temperaturas: T1 = 5.0000, T2 = 5.0000, T3 = 5.0000
Resultado esperado (manual): T1 = T2 = T3 = 5
